# SpendDNA – Minor Project

## Wallet Story Analysis Using Python

**Student Name:** Pradhapkumar S  
**Project Type:** Minor Project2
**Tool Used:** Python, Pandas, NumPy  
**Dataset:** Rahul transaction dataset  

---

## Project Introduction

This project is based on a transaction dataset of Rahul Sharma.  
The dataset contains six months of bank and UPI transaction records.  
The aim of this project is to study Rahul's spending behaviour by cleaning the transaction data, identifying vendors, grouping expenses into categories, and finding patterns in his spending.

In this first part of the project, the main focus is on reading the dataset and cleaning the important columns such as date, amount and transaction type. This is necessary because the raw dataset contains different formats and duplicate rows.

In [64]:
import pandas as pd
import numpy as np

print("Libraries imported successfully")

Libraries imported successfully


## Loading the Dataset

In this step, the CSV file is loaded into a pandas DataFrame.  
After loading, the structure of the dataset is checked using head(), shape and info().

In [65]:
# load dataset
df = pd.read_csv("Data set for DADS June.csv")

print("Dataset loaded successfully")
print("Shape of dataset:", df.shape)

Dataset loaded successfully
Shape of dataset: (1328, 8)


In [66]:
# first 5 rows
df.head()

,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [67]:
# column names
print(df.columns)

Index(['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode',
       'Ref'],
      dtype='object')


In [68]:
# dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1328 entries, 0 to 1327
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         1328 non-null   object 
 1   Time         1328 non-null   object 
 2   Description  1328 non-null   object 
 3   Type         1328 non-null   object 
 4   Amount       1328 non-null   object 
 5   Balance      1328 non-null   float64
 6   Mode         1328 non-null   object 
 7   Ref          1328 non-null   object 
dtypes: float64(1), object(7)
memory usage: 83.1+ KB


In [69]:
# missing values
df.isnull().sum()

,0
Date,0
Time,0
Description,0
Type,0
Amount,0
Balance,0
Mode,0
Ref,0


## Data Cleaning

The raw transaction file contains some formatting issues.  
So in this step, the following cleaning work is done:

- convert Date column into datetime format
- clean Amount column and convert it into numeric
- standardise Type column into debit and credit
- remove duplicate rows

A copy of the original dataset is created before cleaning.

In [70]:
# create copy of dataset
df_clean = df.copy()

print("Copy created")

Copy created


## Cleaning the Date Column

The Date column contains transaction dates in different formats.  
To make analysis easier, the Date column is converted into datetime format using pandas.

In [71]:
# convert Date column into datetime
df_clean["Date"] = pd.to_datetime(df_clean["Date"], errors="coerce", dayfirst=True)

print("Date column converted")
print("Invalid dates:", df_clean["Date"].isnull().sum())

Date column converted
Invalid dates: 1184


## Cleaning the Amount Column

The Amount column contains values in text format with symbols like ₹, Rs. and commas.  
These symbols are removed and the column is converted into numeric format.

In [72]:
# convert amount column to string first
df_clean["Amount"] = df_clean["Amount"].astype(str)

# remove symbols and commas
df_clean["Amount"] = df_clean["Amount"].str.replace("₹", "", regex=False)
df_clean["Amount"] = df_clean["Amount"].str.replace("Rs.", "", regex=False)
df_clean["Amount"] = df_clean["Amount"].str.replace("Rs", "", regex=False)
df_clean["Amount"] = df_clean["Amount"].str.replace(",", "", regex=False)
df_clean["Amount"] = df_clean["Amount"].str.strip()

# convert to numeric
df_clean["Amount"] = pd.to_numeric(df_clean["Amount"], errors="coerce")

print("Amount column cleaned")
print("Invalid amount values:", df_clean["Amount"].isnull().sum())

Amount column cleaned
Invalid amount values: 0


## Standardising the Type Column

The Type column contains values like DR, CR, Debit and Credit.  
For easier analysis, all values are converted into only two standard forms:

- debit
- credit

In [73]:
# convert type column into lowercase
df_clean["Type"] = df_clean["Type"].astype(str).str.lower().str.strip()

# replace values
df_clean["Type"] = df_clean["Type"].replace("dr", "debit")
df_clean["Type"] = df_clean["Type"].replace("cr", "credit")

print("Type column standardised")
print(df_clean["Type"].value_counts())

Type column standardised
Type
debit     1322
credit       6
Name: count, dtype: int64


## Removing Missing Values and Duplicates

After cleaning the main columns, invalid rows and duplicate rows are removed.  
Rows with missing Date or Amount values are dropped because they are important for analysis.

In [74]:
# remove rows where Date or Amount is missing
df_clean = df_clean.dropna(subset=["Date", "Amount"])

# count duplicates
duplicate_count = df_clean.duplicated().sum()

# remove duplicates
df_clean = df_clean.drop_duplicates()

print("Duplicates removed:", duplicate_count)
print("Shape after cleaning:", df_clean.shape)

Duplicates removed: 1
Shape after cleaning: (143, 8)


In [75]:
# preview cleaned dataset
df_clean.head()

,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,debit,2462.0,678275.0,UPI,TXN190872
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,debit,1828.0,-748745.0,UPI,TXN569226
5,2024-01-01,14:48,BHIM ZEPTO,debit,625.0,677650.0,UPI,TXN370902
6,2024-01-01,14:50,UPI-UBER-2426@HDFCBANK,debit,148.0,677020.0,UPI,TXN546173
8,2024-02-01,05:18,POS SWIGGY BANGALORE,debit,537.0,676177.0,UPI,TXN293319


In [76]:
# final info of cleaned data
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 143 entries, 0 to 1197
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         143 non-null    datetime64[ns]
 1   Time         143 non-null    object        
 2   Description  143 non-null    object        
 3   Type         143 non-null    object        
 4   Amount       143 non-null    float64       
 5   Balance      143 non-null    float64       
 6   Mode         143 non-null    object        
 7   Ref          143 non-null    object        
dtypes: datetime64[ns](1), float64(2), object(5)
memory usage: 14.1+ KB


## Conclusion of Part 1

In Part 1, the raw transaction dataset was loaded and cleaned.  
The Date column was converted into datetime format, the Amount column was converted into numeric format, and the Type column was standardised into debit and credit. Duplicate rows and invalid records were also removed.

This cleaned dataset will be used in the next part of the project for vendor extraction and category mapping.

# Part 2 – Vendor Extraction and Category Mapping

In this part of the project, the cleaned transaction data is further processed to identify the actual vendor behind each transaction.  
The **Description** column in the dataset contains raw text such as UPI transaction details, merchant names, ATM withdrawal details and person-to-person transfers.

This raw description is not directly useful for analysis.  
So in this part, the following work is done:

- identify the vendor name from the Description column
- standardise different description formats into one vendor name
- handle special cases like ATM withdrawal and person-to-person transfer
- assign a spending category to each vendor

This step is important because later analysis such as top vendors, top categories, trends and archetypes depends on correct vendor and category mapping.

In [77]:
# check some transaction descriptions
df_clean["Description"].head(20)

,Description
0,AMAZON SELLER SVCS
3,UPI-AMAN-8934@OKAXIS
5,BHIM ZEPTO
6,UPI-UBER-2426@HDFCBANK
8,POS SWIGGY BANGALORE
16,ANI Technologies
20,POS SWIGGY-RESTAURANT
22,POS UBER BANGALORE
23,POS SWIGGY-RESTAURANT
25,TWC INDIA


In [78]:
# vendor dictionary
vendor_dict = {
    "Swiggy": ["SWIGGY", "BUNDL"],
    "Zomato": ["ZOMATO"],
    "Zepto": ["ZEPTO"],
    "Blinkit": ["BLINKIT"],
    "Instamart": ["INSTAMART"],
    "Amazon": ["AMAZON", "AMZN"],
    "Flipkart": ["FLIPKART"],
    "Myntra": ["MYNTRA"],
    "Nykaa": ["NYKAA"],
    "Uber": ["UBER"],
    "Ola": ["OLA"],
    "Rapido": ["RAPIDO"],
    "BookMyShow": ["BOOKMYSHOW"],
    "Netflix": ["NETFLIX"],
    "Spotify": ["SPOTIFY"],
    "YouTube": ["YOUTUBE"],
    "Hotstar": ["HOTSTAR"],
    "Jio": ["JIO"],
    "Airtel": ["AIRTEL"],
    "ACT": ["ACT"],
    "BESCOM": ["BESCOM"],
    "Zerodha": ["ZERODHA"],
    "BigBasket": ["BIGBASKET"],
    "DMart": ["DMART"],
    "Starbucks": ["STARBUCKS"],
    "Third Wave Coffee": ["THIRDWAVE", "THIRD WAVE"],
    "Cafe Coffee Day": ["CCD", "CAFE COFFEE DAY"],
    "Shell": ["SHELL"],
    "Indian Oil": ["INDIANOIL", "INDIAN OIL"],
    "HP": ["HPCL", "HP PETROL"],
    "Rent": ["HOUSE RENT", "RENT"]
}

print("Vendor dictionary created")

Vendor dictionary created


## Vendor Extraction Logic

The Description column contains raw text, so a simple rule-based approach is used to extract vendor names.

### Logic used:
1. Convert description into uppercase text for easy matching.
2. Check if it is an ATM withdrawal.
3. Check if it looks like a person-to-person UPI transfer.
4. Otherwise, search for known vendor keywords from the vendor dictionary.
5. If no match is found, mark it as **Other**.

This is a simple keyword-based vendor extraction method.

In [79]:
def get_vendor(description):
    text = str(description).upper()

    # ATM withdrawal
    if "ATM" in text:
        return "Cash Withdrawal"

    # person to person transfer
    if "UPI-" in text and "@" in text:
        if ("SWIGGY" not in text and "ZOMATO" not in text and "AMAZON" not in text
            and "ZERODHA" not in text and "ZEPTO" not in text and "BLINKIT" not in text):
            return "P2P Transfer"

    # vendor matching from dictionary
    for vendor, keywords in vendor_dict.items():
        for word in keywords:
            if word in text:
                return vendor

    return "Other"

In [80]:
df_clean["vendor_clean"] = df_clean["Description"].apply(get_vendor)

print("Vendor extraction completed")

Vendor extraction completed


In [81]:
df_clean["vendor_clean"].value_counts().head(20)

,count
vendor_clean,
P2P Transfer,28
Swiggy,28
Other,24
Zomato,16
Ola,8
Uber,6
Zepto,5
Zerodha,4
Amazon,3


In [82]:
print("Number of unique vendors:", df_clean["vendor_clean"].nunique())

Number of unique vendors: 20


## Category Mapping

After extracting vendors, the next step is to assign a category to each vendor.  
This makes spending analysis easier because instead of looking at only individual vendors, we can group similar spending into categories such as:

- Food Delivery
- Quick Commerce
- E-commerce
- Transport
- Subscriptions
- Utilities
- Investments
- Groceries
- Cafe
- Fuel
- Entertainment
- Personal Transfer
- Cash Withdrawal

A simple dictionary is used to map each vendor to its category.

In [83]:
category_dict = {
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",
    "Instamart": "Quick Commerce",

    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",

    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",

    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "YouTube": "Subscriptions",
    "Hotstar": "Subscriptions",

    "Jio": "Utilities",
    "Airtel": "Utilities",
    "ACT": "Utilities",
    "BESCOM": "Utilities",

    "Zerodha": "Investments",

    "BigBasket": "Groceries",
    "DMart": "Groceries",

    "Starbucks": "Cafe",
    "Third Wave Coffee": "Cafe",
    "Cafe Coffee Day": "Cafe",

    "Shell": "Fuel",
    "Indian Oil": "Fuel",
    "HP": "Fuel",

    "BookMyShow": "Entertainment",

    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal",

    "Rent": "Rent"
}

In [84]:
df_clean["category"] = df_clean["vendor_clean"].map(category_dict)

# if vendor not found in category dictionary
df_clean["category"] = df_clean["category"].fillna("Other")

print("Category mapping completed")

Category mapping completed


In [85]:
df_clean["category"].value_counts()

,count
category,
Food Delivery,44
Personal Transfer,28
Other,24
Transport,15
Quick Commerce,10
E-commerce,9
Investments,4
Cash Withdrawal,3
Rent,2


In [86]:
df_clean[["Date", "Description", "vendor_clean", "category", "Amount"]].head(20)

,Date,Description,vendor_clean,category,Amount
0,2024-01-01,AMAZON SELLER SVCS,Amazon,E-commerce,2462.0
3,2024-01-01,UPI-AMAN-8934@OKAXIS,P2P Transfer,Personal Transfer,1828.0
5,2024-01-01,BHIM ZEPTO,Zepto,Quick Commerce,625.0
6,2024-01-01,UPI-UBER-2426@HDFCBANK,P2P Transfer,Personal Transfer,148.0
8,2024-02-01,POS SWIGGY BANGALORE,Swiggy,Food Delivery,537.0
16,2024-02-01,ANI Technologies,Other,Other,267.0
20,2024-03-01,POS SWIGGY-RESTAURANT,Swiggy,Food Delivery,537.0
22,2024-03-01,POS UBER BANGALORE,Uber,Transport,439.0
23,2024-03-01,POS SWIGGY-RESTAURANT,Swiggy,Food Delivery,238.0
25,2024-04-01,TWC INDIA,Other,Other,272.0


In [87]:
df_clean["vendor_clean"].value_counts().head(30)

,count
vendor_clean,
P2P Transfer,28
Swiggy,28
Other,24
Zomato,16
Ola,8
Uber,6
Zepto,5
Zerodha,4
Amazon,3


In [88]:
df_clean[df_clean["vendor_clean"] == "Other"]["Description"].unique()[:50]

array(['ANI Technologies', 'TWC INDIA', 'BANGALORE ELEC SUPPLY',
       'POS EMPIRE RESTAURANT', 'ROPPEN TRANSPORTATION',
       'NEFT-TECHCRUSH LABS-SALARY MAY24', 'COFFEE DAY GLOBAL',
       'BMS MOVIE TICKETS', 'INNOVATIVE RETAIL', 'AVENUE SUPERMARTS',
       'KIRANAKART TECH', 'POS BANGALORE RESTAURANT'], dtype=object)

In [89]:
df_clean["category"].value_counts()

,count
category,
Food Delivery,44
Personal Transfer,28
Other,24
Transport,15
Quick Commerce,10
E-commerce,9
Investments,4
Cash Withdrawal,3
Rent,2


## Conclusion of Part 2

In this part of the project, vendor names were extracted from the raw transaction descriptions using a simple keyword-based method.  
Special cases such as ATM withdrawals and person-to-person UPI transfers were also handled separately. After that, each vendor was assigned to a spending category using a category mapping dictionary.

This step makes the transaction data more meaningful and ready for the main spending analysis. In the next part of the project, the cleaned and categorised data will be used to calculate total spending, top categories, top vendors and monthly spending patterns.

# Part 3 – Spending Overview and Monthly Trend Analysis

After cleaning the dataset, extracting vendors and assigning categories, the next step is to analyse Rahul's spending behaviour.

In this part of the project, the main focus is on answering basic financial questions such as:

- How much total money came into the account?
- How much total money was spent?
- What is the net savings or net change?
- Which categories received the highest spending?
- Which vendors received the highest spending?
- How did the spending change from month to month?

This section gives a summary of Rahul's overall financial activity and prepares the base for deeper pattern analysis in the next part.

In [90]:
# separate debit and credit transactions
debit_df = df_clean[df_clean["Type"] == "debit"]
credit_df = df_clean[df_clean["Type"] == "credit"]

print("Debit transactions:", debit_df.shape[0])
print("Credit transactions:", credit_df.shape[0])

Debit transactions: 141
Credit transactions: 2


## Overall Spending Summary

To understand Rahul's financial activity, the total credits and total debits are calculated separately.

- **Total Credit** means money received into the account.
- **Total Debit** means money spent from the account.

Using these values, the net change in money can also be calculated.

In [91]:
# total credit and debit
total_credit = credit_df["Amount"].sum()
total_debit = debit_df["Amount"].sum()

net_change = total_credit - total_debit

print("Total Credit :", total_credit)
print("Total Debit  :", total_debit)
print("Net Change   :", net_change)

Total Credit : 170094.0
Total Debit  : 205310.0
Net Change   : -35216.0


In [92]:
print("========== SPENDING OVERVIEW ==========")
print("Total Credit :", round(total_credit, 2))
print("Total Debit  :", round(total_debit, 2))
print("Net Change   :", round(net_change, 2))
print("=======================================")

========== SPENDING OVERVIEW ==========
Total Credit : 170094.0
Total Debit  : 205310.0
Net Change   : -35216.0


## Category-wise Spending Analysis

After finding the overall spending, the next step is to understand **where the money went**.

For this, debit transactions are grouped by **category** and the total amount spent in each category is calculated.  
This helps identify which spending areas such as Food Delivery, Quick Commerce or E-commerce contribute the most to Rahul's expenses.

In [93]:
# total spending by category
category_spend = debit_df.groupby("category")["Amount"].sum().sort_values(ascending=False)

category_spend

,Amount
category,
Investments,60000.0
Personal Transfer,38070.0
Rent,36000.0
E-commerce,20362.0
Food Delivery,19617.0
Other,16411.0
Transport,4962.0
Quick Commerce,4782.0
Cash Withdrawal,3000.0


In [94]:
top_5_categories = category_spend.head(5)

print("Top 5 Spending Categories")
print(top_5_categories)

Top 5 Spending Categories
category
Investments          60000.0
Personal Transfer    38070.0
Rent                 36000.0
E-commerce           20362.0
Food Delivery        19617.0
Name: Amount, dtype: float64


In [95]:
# percentage contribution of each category
category_percent = (category_spend / total_debit) * 100
category_percent = category_percent.sort_values(ascending=False)

category_percent.head(10)

,Amount
category,
Investments,29.224100
Personal Transfer,18.542692
Rent,17.534460
E-commerce,9.917685
Food Delivery,9.554820
Other,7.993278
Transport,2.416833
Quick Commerce,2.329161
Cash Withdrawal,1.461205


## Vendor-wise Spending Analysis

Category analysis gives a broad picture, but vendor analysis gives more detail.  
In this step, debit transactions are grouped by **vendor_clean** and the total amount spent on each vendor is calculated.

This helps identify the merchants or platforms where Rahul spends the most money.

In [96]:
# total spending by vendor
vendor_spend = debit_df.groupby("vendor_clean")["Amount"].sum().sort_values(ascending=False)

vendor_spend.head(10)

,Amount
vendor_clean,
Zerodha,60000.0
P2P Transfer,38070.0
Rent,36000.0
Other,16411.0
Swiggy,12638.0
Flipkart,8458.0
Amazon,7550.0
Zomato,6979.0
Nykaa,4354.0


In [97]:
print("Top 10 Vendors by Spending")
print(vendor_spend.head(10))

Top 10 Vendors by Spending
vendor_clean
Zerodha            60000.0
P2P Transfer       38070.0
Rent               36000.0
Other              16411.0
Swiggy             12638.0
Flipkart            8458.0
Amazon              7550.0
Zomato              6979.0
Nykaa               4354.0
Cash Withdrawal     3000.0
Name: Amount, dtype: float64


## Monthly Spending Trend

To understand whether spending increased or decreased over time, monthly spending analysis is performed.

In this step:
- the month is taken from the Date column
- debit transactions are grouped month-wise
- total spending for each month is calculated

This helps in understanding how Rahul's expenses changed from January to June.

In [98]:
# create month column if not already present
df_clean["month"] = df_clean["Date"].dt.month
debit_df["month"] = debit_df["Date"].dt.month

print("Month column ready")

Month column ready


/tmp/ipykernel_3849/2773244562.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  debit_df["month"] = debit_df["Date"].dt.month


In [99]:
# total monthly spending
monthly_spend = debit_df.groupby("month")["Amount"].sum().sort_index()

monthly_spend

,Amount
month,
1,14972.0
2,6390.0
3,26578.0
4,5246.0
5,21011.0
6,9292.0
7,23632.0
8,41383.0
9,14154.0


In [100]:
print("Monthly Spending Trend")
print(monthly_spend)

Monthly Spending Trend
month
1     14972.0
2      6390.0
3     26578.0
4      5246.0
5     21011.0
6      9292.0
7     23632.0
8     41383.0
9     14154.0
10     5541.0
11     6366.0
12    30745.0
Name: Amount, dtype: float64


## Category-wise Monthly Spending

To go one step deeper, monthly spending can also be analysed category-wise.  
This shows how much was spent in each category in each month.

A pivot table is used here because it gives a simple table format where:
- rows represent categories
- columns represent months
- values represent total spending

In [101]:
# category-wise monthly spending table
monthly_category_table = debit_df.pivot_table(
    values="Amount",
    index="category",
    columns="month",
    aggfunc="sum"
)

monthly_category_table

month,1,2,3,4,5,6,7,8,9,10,11,12
category,,,,,,,,,,,,
Cafe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,809.0,NaN
Cash Withdrawal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2500.0,500.0
E-commerce,4088.0,996.0,2147.0,3311.0,NaN,4868.0,NaN,860.0,NaN,NaN,NaN,4092.0
Food Delivery,1180.0,3454.0,2862.0,632.0,1159.0,1636.0,1686.0,1446.0,712.0,183.0,1909.0,2758.0
Fuel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1087.0
Investments,NaN,NaN,NaN,NaN,NaN,NaN,15000.0,30000.0,NaN,NaN,NaN,15000.0
Other,2255.0,267.0,1416.0,513.0,753.0,794.0,368.0,1762.0,421.0,1964.0,766.0,5132.0
Personal Transfer,5155.0,355.0,1714.0,790.0,166.0,1860.0,5717.0,6949.0,10991.0,2162.0,35.0,2176.0
Quick Commerce,1616.0,697.0,NaN,NaN,502.0,NaN,510.0,NaN,1110.0,NaN,347.0,NaN


In [102]:
monthly_category_table = monthly_category_table.fillna(0)
monthly_category_table

month,1,2,3,4,5,6,7,8,9,10,11,12
category,,,,,,,,,,,,
Cafe,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,809.0,0.0
Cash Withdrawal,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2500.0,500.0
E-commerce,4088.0,996.0,2147.0,3311.0,0.0,4868.0,0.0,860.0,0.0,0.0,0.0,4092.0
Food Delivery,1180.0,3454.0,2862.0,632.0,1159.0,1636.0,1686.0,1446.0,712.0,183.0,1909.0,2758.0
Fuel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1087.0
Investments,0.0,0.0,0.0,0.0,0.0,0.0,15000.0,30000.0,0.0,0.0,0.0,15000.0
Other,2255.0,267.0,1416.0,513.0,753.0,794.0,368.0,1762.0,421.0,1964.0,766.0,5132.0
Personal Transfer,5155.0,355.0,1714.0,790.0,166.0,1860.0,5717.0,6949.0,10991.0,2162.0,35.0,2176.0
Quick Commerce,1616.0,697.0,0.0,0.0,502.0,0.0,510.0,0.0,1110.0,0.0,347.0,0.0


In [103]:
highest_month = monthly_spend.idxmax()
highest_month_value = monthly_spend.max()

lowest_month = monthly_spend.idxmin()
lowest_month_value = monthly_spend.min()

print("Highest spending month:", highest_month, "-", highest_month_value)
print("Lowest spending month :", lowest_month, "-", lowest_month_value)

Highest spending month: 8 - 41383.0
Lowest spending month : 4 - 5246.0


## Basic Observations from Part 3

From the spending overview and monthly trend analysis, the following basic observations can be made:

- the total debit amount shows Rahul's overall spending across six months
- category-wise totals show the areas where most money was spent
- vendor-wise totals show the platforms or merchants used most often
- monthly spending trend shows whether spending stayed stable or changed over time

These results form the base for the next stage of analysis, where time-of-day patterns, anomalies and spending archetypes will be identified.

In [104]:
print("============== PART 3 SUMMARY ==============")
print("Total Credit :", round(total_credit, 2))
print("Total Debit  :", round(total_debit, 2))
print("Net Change   :", round(net_change, 2))
print()

print("Top 5 Categories")
print(top_5_categories)
print()

print("Top 5 Vendors")
print(vendor_spend.head(5))
print()

print("Monthly Spending")
print(monthly_spend)
print("============================================")

============== PART 3 SUMMARY ==============
Total Credit : 170094.0
Total Debit  : 205310.0
Net Change   : -35216.0

Top 5 Categories
category
Investments          60000.0
Personal Transfer    38070.0
Rent                 36000.0
E-commerce           20362.0
Food Delivery        19617.0
Name: Amount, dtype: float64

Top 5 Vendors
vendor_clean
Zerodha         60000.0
P2P Transfer    38070.0
Rent            36000.0
Other           16411.0
Swiggy          12638.0
Name: Amount, dtype: float64

Monthly Spending
month
1     14972.0
2      6390.0
3     26578.0
4      5246.0
5     21011.0
6      9292.0
7     23632.0
8     41383.0
9     14154.0
10     5541.0
11     6366.0
12    30745.0
Name: Amount, dtype: float64


## Conclusion of Part 3

In Part 3, the overall financial activity of Rahul Sharma was analysed using the cleaned and categorised transaction data. Total credits, total debits and net change were calculated to understand the broad financial picture. After that, spending was grouped by category and vendor to identify the major spending areas and the most frequently used merchants.

Finally, monthly spending analysis was performed to understand how Rahul's expenses changed over the six-month period. This part provides the core summary of spending behaviour and acts as the foundation for the next stage of analysis.

In [105]:
debit_df["month"] = debit_df["Date"].dt.month

/tmp/ipykernel_3849/1131875435.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  debit_df["month"] = debit_df["Date"].dt.month


# Part 4 – Time Pattern Analysis, Anomaly Detection and Final Report

In the previous parts of the project, the transaction data was cleaned, vendors were identified, categories were assigned and the overall spending pattern was analysed.  

In this final part, the analysis is extended further to study:

- the time of day when Rahul spends the most
- transactions that are unusually high compared to normal spending
- Rahul's spending personality or spending archetypes
- a final printed SpendDNA report summarising the complete analysis

This part helps convert the transaction data into a more meaningful story rather than just a list of totals.

## Time-of-Day Analysis

Spending behaviour is not only about **how much** money is spent, but also **when** it is spent.  
For example, food delivery spending late at night may show a different lifestyle pattern compared to morning cafe spending.

To study this, the transaction hour is extracted from the **Time** column and spending is grouped by hour.

In [106]:
# create hour column from Time
df_clean["hour"] = df_clean["Time"].astype(str).str[:2]
df_clean["hour"] = pd.to_numeric(df_clean["hour"], errors="coerce")

# create debit dataframe again to make sure it includes hour column
debit_df = df_clean[df_clean["Type"] == "debit"].copy()

print("Hour column created")

Hour column created


In [107]:
# total spending by hour
hourly_spend = debit_df.groupby("hour")["Amount"].sum().sort_index()

hourly_spend

,Amount
hour,
0,1820.0
1,1462.0
2,1015.0
3,3228.0
4,15413.0
5,5458.0
6,1948.0
7,1668.0
8,3542.0


In [108]:
print("Hourly Spending")
print(hourly_spend)

Hourly Spending
hour
0      1820.0
1      1462.0
2      1015.0
3      3228.0
4     15413.0
5      5458.0
6      1948.0
7      1668.0
8      3542.0
9      2678.0
10    49574.0
11     8739.0
12     3729.0
13    22872.0
14    23670.0
15     3765.0
16     2753.0
17     3200.0
18    14213.0
19     5950.0
20    16097.0
21     7361.0
22     3728.0
23     1427.0
Name: Amount, dtype: float64


In [109]:
# food delivery transactions
food_df = debit_df[debit_df["category"] == "Food Delivery"]

# total food delivery transactions
total_food_orders = food_df.shape[0]

# late-night food orders (9 PM to 2 AM)
late_night_food = food_df[(food_df["hour"] >= 21) | (food_df["hour"] <= 2)]

late_night_count = late_night_food.shape[0]

if total_food_orders > 0:
    late_night_percent = (late_night_count / total_food_orders) * 100
else:
    late_night_percent = 0

print("Total food delivery transactions:", total_food_orders)
print("Late-night food delivery transactions:", late_night_count)
print("Late-night food delivery percentage:", round(late_night_percent, 2), "%")

Total food delivery transactions: 44
Late-night food delivery transactions: 11
Late-night food delivery percentage: 25.0 %


## Anomaly Detection

An anomaly is a transaction whose amount is unusually high compared to other transactions in the same category.

In this project, anomaly detection is done using a simple **z-score method**:

\[
z = \frac{value - mean}{standard\ deviation}
\]

If the z-score is high, it means the transaction amount is much larger than the normal spending pattern of that category.

In [110]:
# mean amount by category
category_mean = debit_df.groupby("category")["Amount"].transform("mean")

# standard deviation by category
category_std = debit_df.groupby("category")["Amount"].transform("std")

# z-score
debit_df["z_score"] = (debit_df["Amount"] - category_mean) / category_std

print("Z-score column created")

Z-score column created


In [111]:
# anomaly transactions
anomalies = debit_df[debit_df["z_score"] > 2]

print("Number of anomalies:", anomalies.shape[0])

Number of anomalies: 3


In [112]:
anomalies[["Date", "vendor_clean", "category", "Amount", "z_score"]].sort_values(
    by="z_score", ascending=False
).head(10)

,Date,vendor_clean,category,Amount,z_score
70,2024-09-01,P2P Transfer,Personal Transfer,10745.0,4.521389
521,2024-12-03,Other,Other,2430.0,2.375388
85,2024-12-01,Other,Other,2329.0,2.232925


## Spending Archetype Detection

A spending archetype is a simple label used to describe Rahul's financial behaviour based on the category and timing of his transactions.

The archetypes used in this project are based on the project brief.  
Examples include:

- **The Foodie**
- **The Quick Commerce Junkie**
- **The Shopaholic**
- **The Investor**
- **The Late-Night Snacker**
- **The YOLO Spender**

These labels are assigned using simple rule-based conditions.

In [113]:
# category spending percentages
food_spend = category_spend.get("Food Delivery", 0)
restaurant_spend = category_spend.get("Restaurants", 0)
cafe_spend = category_spend.get("Cafe", 0)
quick_spend = category_spend.get("Quick Commerce", 0)
ecommerce_spend = category_spend.get("E-commerce", 0)
investment_spend = category_spend.get("Investments", 0)
transport_spend = category_spend.get("Transport", 0)
subscription_spend = category_spend.get("Subscriptions", 0)

food_total = food_spend + restaurant_spend + cafe_spend

food_percent = (food_total / total_debit) * 100 if total_debit > 0 else 0
quick_percent = (quick_spend / total_debit) * 100 if total_debit > 0 else 0
ecommerce_percent = (ecommerce_spend / total_debit) * 100 if total_debit > 0 else 0
investment_percent = (investment_spend / total_debit) * 100 if total_debit > 0 else 0
transport_percent = (transport_spend / total_debit) * 100 if total_debit > 0 else 0
subscription_percent = (subscription_spend / total_debit) * 100 if total_debit > 0 else 0

# savings rate
savings_rate = ((total_credit - total_debit) / total_credit) * 100 if total_credit > 0 else 0

print("Food %:", round(food_percent, 2))
print("Quick Commerce %:", round(quick_percent, 2))
print("E-commerce %:", round(ecommerce_percent, 2))
print("Investment %:", round(investment_percent, 2))
print("Savings Rate %:", round(savings_rate, 2))

Food %: 9.95
Quick Commerce %: 2.33
E-commerce %: 9.92
Investment %: 29.22
Savings Rate %: -20.7


In [114]:
archetypes = []

# foodie
if food_percent > 25:
    archetypes.append("The Foodie")

# quick commerce junkie
if quick_percent > 15:
    archetypes.append("The Quick Commerce Junkie")

# shopaholic
if ecommerce_percent > 15:
    archetypes.append("The Shopaholic")

# investor
if investment_percent > 15:
    archetypes.append("The Investor")

# late-night snacker
if late_night_percent > 50:
    archetypes.append("The Late-Night Snacker")

# cab commuter
if transport_percent > 10:
    archetypes.append("The Cab Commuter")

# yolo spender
if savings_rate < 10:
    archetypes.append("The YOLO Spender")

# disciplined saver
if savings_rate > 40:
    archetypes.append("The Disciplined Saver")

print("Detected Archetypes:")
for item in archetypes:
    print("-", item)

Detected Archetypes:
- The Investor
- The YOLO Spender


## Final SpendDNA Report

In the final step, the main results from the project are brought together into one printed report.  
This report includes:

- total credits and debits
- net change
- top spending categories
- top vendors
- monthly spending
- late-night food behaviour
- anomaly count
- detected spending archetypes

This acts as the final summary of Rahul's six-month spending story.

In [115]:
print("=" * 60)
print("SPENDDNA REPORT - RAHUL SHARMA")
print("=" * 60)

print("\n1. OVERALL SUMMARY")
print("Total Credit :", round(total_credit, 2))
print("Total Debit  :", round(total_debit, 2))
print("Net Change   :", round(net_change, 2))
print("Savings Rate :", round(savings_rate, 2), "%")

print("\n2. TOP 5 CATEGORIES")
print(top_5_categories)

print("\n3. TOP 5 VENDORS")
print(vendor_spend.head(5))

print("\n4. MONTHLY SPENDING")
print(monthly_spend)

print("\n5. LATE-NIGHT FOOD ANALYSIS")
print("Late-night food delivery percentage:", round(late_night_percent, 2), "%")

print("\n6. ANOMALY SUMMARY")
print("Total anomalies found:", anomalies.shape[0])

print("\n7. SPENDING ARCHETYPES")
for item in archetypes:
    print("-", item)

print("=" * 60)

SPENDDNA REPORT - RAHUL SHARMA

1. OVERALL SUMMARY
Total Credit : 170094.0
Total Debit  : 205310.0
Net Change   : -35216.0
Savings Rate : -20.7 %

2. TOP 5 CATEGORIES
category
Investments          60000.0
Personal Transfer    38070.0
Rent                 36000.0
E-commerce           20362.0
Food Delivery        19617.0
Name: Amount, dtype: float64

3. TOP 5 VENDORS
vendor_clean
Zerodha         60000.0
P2P Transfer    38070.0
Rent            36000.0
Other           16411.0
Swiggy          12638.0
Name: Amount, dtype: float64

4. MONTHLY SPENDING
month
1     14972.0
2      6390.0
3     26578.0
4      5246.0
5     21011.0
6      9292.0
7     23632.0
8     41383.0
9     14154.0
10     5541.0
11     6366.0
12    30745.0
Name: Amount, dtype: float64

5. LATE-NIGHT FOOD ANALYSIS
Late-night food delivery percentage: 25.0 %

6. ANOMALY SUMMARY
Total anomalies found: 3

7. SPENDING ARCHETYPES
- The Investor
- The YOLO Spender


## Key Insights

Based on the analysis, a few simple insights can be written from Rahul's transaction history:

1. Rahul's debit amount is higher than his credit amount, which indicates a negative savings pattern over the six-month period.
2. A large part of his spending is concentrated in a few categories such as food delivery, quick commerce and e-commerce.
3. A significant percentage of food delivery transactions happen late at night, which suggests a repeated time-based spending habit.
4. The anomaly analysis shows that a small number of transactions are unusually large compared to Rahul's normal category-level spending.
5. The archetype analysis suggests that Rahul's financial behaviour is not limited to one pattern but is a combination of multiple spending habits.

# Final Conclusion

This project helped me understand how raw transaction data can be converted into meaningful financial insights using Python, Pandas and NumPy. The dataset initially contained raw descriptions, mixed formats and duplicate records, so the first part of the work focused on cleaning and standardising the data.

After cleaning, vendor extraction and category mapping were performed to make the transaction data more useful for analysis. Once the data was prepared, I analysed Rahul Sharma's spending by looking at category-wise totals, vendor-wise totals, monthly spending trends, time-of-day behaviour and unusual high-value transactions.

The final stage of the project focused on identifying spending archetypes and presenting the complete result as a simple SpendDNA report. Overall, this project gave me practical experience in data cleaning, grouping, aggregation, transaction analysis and basic behaviour-based insight generation using Python.

## AI Assistance Disclosure

I used AI support only for learning help, debugging support and understanding the project structure.  
The final notebook structure, cleaning logic, mapping logic, analysis flow and outputs were prepared, checked and organised by me based on the given dataset and project brief.

# Part 5 – Final Insights, Limitations and Future Scope

In the previous sections of the project, the transaction data was cleaned, vendors were extracted, spending categories were assigned and Rahul Sharma's financial behaviour was analysed using category trends, vendor patterns, time-of-day analysis, anomaly detection and spending archetypes.

In this final section, the project is concluded by summarising the key findings, discussing the limitations of the current analysis and suggesting possible future improvements. This helps present the project not only as a coding exercise, but also as a practical financial behaviour analysis case study.

## Final Insights from the SpendDNA Analysis

Based on the six-month transaction analysis, the following important insights can be observed from Rahul Sharma's spending behaviour:

### 1. Spending pattern is concentrated in a few categories
A major portion of Rahul's total debit amount is concentrated in a limited number of categories such as food delivery, quick commerce, e-commerce and transport. This shows that a few lifestyle-driven spending areas contribute heavily to his monthly expenses.

### 2. Vendor-level analysis gives more detail than raw transaction data
The original transaction descriptions were messy and difficult to understand directly. After vendor extraction and category mapping, the same data became much more meaningful. This shows the importance of cleaning and standardising financial data before analysis.

### 3. Monthly spending is not constant
The month-wise spending trend shows that Rahul's expenses change from month to month instead of staying fixed. This variation may be caused by shopping spikes, travel, investment activity, bill payments or lifestyle-related purchases in certain months.

### 4. Time-based analysis reveals behavioural habits
The hour-based analysis and late-night food delivery analysis help move beyond simple totals and show behavioural patterns. Spending time can sometimes reveal personal habits more clearly than total amount alone.

### 5. Anomalies help identify unusual spending events
The anomaly detection step highlights transactions that are much larger than Rahul's usual category-level spending. Such transactions may represent one-time purchases, exceptional payments or unusual financial behaviour during the six-month period.

### 6. Spending archetypes make the analysis more interpretable
Instead of showing only numbers, the archetype section converts financial patterns into simple behavioural labels such as foodie, investor, shopaholic or late-night snacker. This makes the final analysis easier to understand and more engaging.

## Real-World Usefulness of This Project

This project is useful beyond academic submission because transaction analysis is a real business use case in banking, fintech and personal finance applications.

### Some practical uses of this type of analysis are:

- **personal finance apps** can show category-wise spending and monthly trends to users
- **banks and fintech companies** can identify customer behaviour patterns from transaction data
- **budgeting tools** can help users understand overspending categories
- **anomaly detection systems** can help flag unusual transactions for further checking
- **behaviour-based insights** can be used to build personalised offers or financial recommendations

Therefore, this project demonstrates how data cleaning and analysis can be applied to solve a realistic financial analytics problem.

## Limitations of the Current Project

Although the project gives useful insights, it also has some limitations.

### 1. Vendor extraction is rule-based
Vendor extraction in this notebook is based on keyword matching. If a transaction description contains a new or unusual format that is not present in the vendor dictionary, it may be classified as **Other**.

### 2. Category mapping depends on the vendor dictionary
The category assigned to a transaction depends on the correctness of the vendor mapping. If a vendor is missed or incorrectly labelled, the category analysis may also change.

### 3. Time analysis depends on available transaction time
If the Time column is missing, incomplete or inconsistent, time-of-day analysis may not fully reflect actual spending behaviour.

### 4. Archetypes are based on simple thresholds
The archetype labels are assigned using simple percentage rules. These labels are useful for interpretation, but they should not be treated as exact psychological conclusions.

### 5. The analysis is based on one person's transaction history
This project analyses only Rahul Sharma's six-month dataset. So the results are specific to this dataset and cannot be generalised to all users without larger data.

## Future Improvements

This project can be improved further in the following ways:

### 1. Better vendor extraction
A larger vendor dictionary can be built by studying all unmatched descriptions and adding more merchant keywords. This would reduce the number of transactions classified as **Other**.

### 2. Better category design
Some categories can be split further into smaller groups such as groceries, dining out, medicine, shopping, entertainment and bills for more detailed analysis.

### 3. Visualisations
Graphs such as bar charts, line charts and pie charts can be added to make the monthly trend, top vendors and top categories easier to understand visually.

### 4. Personal budgeting suggestions
The notebook can be extended to provide recommendations such as:
- top overspending categories
- suggested monthly budget
- possible savings opportunities
- subscription clean-up suggestions

### 5. Dashboard version
The project can be converted into an interactive dashboard using tools such as Streamlit or Power BI so that users can upload their own bank statements and view insights automatically.

## Final Project Conclusion

The SpendDNA minor project was developed to study and interpret Rahul Sharma's six-month transaction history using Python-based data analysis. The project began with raw bank and UPI transaction data and transformed it into a structured financial story through cleaning, parsing, vendor extraction, category mapping and behavioural analysis.

The work done in this notebook shows the full data analysis pipeline in a simple but practical form:
- raw data loading
- cleaning and standardisation
- feature creation
- category-level and vendor-level analysis
- monthly trend analysis
- time-based behaviour analysis
- anomaly detection
- archetype identification
- final insight reporting

Overall, this project helped me understand how real-world transaction data can be processed and analysed to generate useful personal finance insights. It also improved my understanding of pandas operations such as filtering, grouping, aggregation, cleaning and rule-based analysis. Most importantly, it showed how a raw CSV file can be converted into a meaningful analytical report.

In [116]:
print("=" * 65)
print("SpendDNA Minor Project Completed Successfully")
print("=" * 65)
print("Project Title : SpendDNA - Wallet Story Analysis")
print("Student Name  : Pradhapkumar S")
print("Tools Used    : Python, Pandas, NumPy")
print("Dataset Used  : Rahul Sharma Transaction Dataset")
print("Status        : Notebook completed")
print("=" * 65)

SpendDNA Minor Project Completed Successfully
Project Title : SpendDNA - Wallet Story Analysis
Student Name  : Pradhapkumar S
Tools Used    : Python, Pandas, NumPy
Dataset Used  : Rahul Sharma Transaction Dataset
Status        : Notebook completed


# Part 6 – Viva Preparation and Submission Support

After completing the SpendDNA notebook, the final step is to prepare for project explanation and submission.  
A minor project is not only about writing code, but also about understanding what was done, why it was done and how to explain the results clearly.

This section contains:
- simple viva questions and answers
- explanation points for each project section
- common errors that may occur while running the notebook
- final checklist before submission

## Viva Question 1 – What is the objective of this project?

### Answer:
The main objective of this project is to analyse Rahul Sharma's six-month transaction history and understand his spending behaviour. The project converts raw bank and UPI transaction data into useful insights by cleaning the data, extracting vendor names, assigning spending categories, analysing monthly spending, identifying unusual transactions and detecting spending archetypes.

## Viva Question 3 – What is vendor extraction?

### Answer:
Vendor extraction means identifying the actual merchant or source of a transaction from the raw Description column. For example, if the description contains a long UPI text with the word Swiggy, the transaction is mapped to the vendor Swiggy. This is useful because raw descriptions are difficult to analyse directly, but vendor names make the transaction data more meaningful.

## Viva Question 4 – How did you extract vendors in this project?

### Answer:
I used a simple keyword-based approach. First, I converted the Description text into uppercase. Then I checked whether the description matched known vendor keywords such as SWIGGY, ZOMATO, AMAZON, UBER or ZEPTO. If a match was found, I assigned that vendor name. If the description contained ATM, I marked it as Cash Withdrawal. If it looked like a person-to-person UPI transfer, I marked it as P2P Transfer. If no keyword matched, I marked it as Other.

## Viva Question 5 – Why did you do category mapping after vendor extraction?

### Answer:
Vendor extraction gives the merchant name, but category mapping groups similar vendors into larger spending categories. For example, Swiggy and Zomato both belong to Food Delivery, while Amazon and Flipkart belong to E-commerce. This helps in analysing category-wise spending instead of looking only at individual transactions.

## Viva Question 6 – What is anomaly detection in this project?

### Answer:
Anomaly detection is used to find transactions whose amount is unusually high compared to Rahul's normal spending pattern. In this project, anomaly detection was done category-wise using the z-score method. If a transaction had a z-score greater than 2, it was treated as an unusual or high-value transaction within that category.

## Viva Question 7 – Why did you calculate anomalies category-wise instead of using all transactions together?

### Answer:
I calculated anomalies category-wise because spending patterns are different for different categories. For example, a ₹2000 Amazon purchase may be normal in E-commerce, but a ₹2000 food delivery order may be unusual in Food Delivery. If all transactions are mixed together, the anomaly detection becomes less meaningful.

## Viva Question 8 – What are spending archetypes?

### Answer:
Spending archetypes are simple behavioural labels assigned based on Rahul's transaction patterns. For example, if a large percentage of spending goes to food delivery, Rahul may be labelled as The Foodie. If investment spending is high, he may be labelled as The Investor. These archetypes make the analysis easier to interpret by converting numbers into simple behavioural patterns.

## Viva Question 9 – What Python concepts did you use in this project?

### Answer:
In this project, I used Python and pandas for:
- reading CSV files
- cleaning string values
- converting columns into datetime and numeric format
- filtering rows
- grouping data using groupby
- calculating sums and percentages
- sorting values
- applying vendor extraction logic
- creating pivot tables for monthly category analysis

## Viva Question 10 – What are the main outputs of this project?

### Answer:
The main outputs of this project are:
- cleaned transaction dataset
- vendor-wise transaction mapping
- category-wise spending analysis
- top vendors and top categories
- monthly spending trend
- time-of-day spending behaviour
- anomaly transaction list
- spending archetype labels
- final SpendDNA report

## How to Explain the Full Project in 1 Minute

### Short Project Explanation:
This project is called SpendDNA and it is based on Rahul Sharma's six-month bank and UPI transaction data. The aim of the project is to analyse his spending behaviour using Python. First, I loaded and cleaned the raw transaction data by standardising the date, amount and transaction type columns and removing duplicates. Then I extracted vendor names from the Description column and mapped those vendors into spending categories such as Food Delivery, E-commerce and Transport. After preparing the data, I analysed total credits, total debits, category-wise spending, vendor-wise spending and monthly spending trends. Finally, I performed time-of-day analysis, anomaly detection and spending archetype detection to understand Rahul's financial behaviour in a more detailed way.

## Common Errors and How to Fix Them

### 1. File not found error
If Colab shows a file not found error while reading the CSV file, check whether the uploaded file name matches the file name used in `pd.read_csv()`.

### 2. Date conversion issue
If the Date column gives many missing values after conversion, check whether the date format in the CSV file is different from what was assumed in the notebook.

### 3. Too many "Other" vendors
If many transactions are marked as Other, it means the vendor dictionary needs more keywords.

### 4. Category totals look wrong
This usually happens when vendor extraction is incomplete or when the Type column is not properly standardised into debit and credit.

### 5. No anomalies found
This may happen if category-wise standard deviation is very small, if some categories have very few transactions or if the threshold is too high.

## Final Submission Checklist

Before submitting the project, I should check the following points:

- the notebook runs from top to bottom without errors
- the dataset file name is correct
- all markdown headings are present
- vendor extraction and category mapping cells are executed
- the final SpendDNA report is printed properly
- the notebook includes conclusion and reflection sections
- the notebook is renamed correctly as required by the project brief
- the Colab sharing setting is changed to **Anyone with the link can view**

## Final Note

This project helped me understand that data analysis is not only about applying functions on a dataset, but also about cleaning messy real-world data, designing meaningful categories and converting raw numbers into understandable insights. The SpendDNA project gave me practical exposure to transaction analysis, financial behaviour analysis and rule-based data processing using Python.

# Part 7 – Project Presentation and Documentation Support

After completing the technical analysis of the SpendDNA project, the next important step is to present the project properly. In academic projects and interviews, it is not enough to only run the notebook. The student should also be able to explain the purpose of the project, the workflow used, the outputs generated and the value of the project in a clear way.

This section contains:
- a short presentation script for the project
- resume points for adding the project in a CV
- project summary text for documentation
- demo explanation points

## 2-Minute Project Presentation Script

### Project Title:
SpendDNA – Wallet Story Analysis Using Python

### Presentation Script:
Good morning. My project is titled **SpendDNA**, and it focuses on analysing Rahul Sharma's six-month transaction history to understand his spending behaviour. The dataset contains bank and UPI transaction records, including date, time, description, transaction type, amount and balance details.

The first step of the project was **data cleaning**. I converted the Date column into datetime format, cleaned the Amount column, standardised the transaction Type column and removed duplicate or invalid records. After that, I worked on **vendor extraction**, where I identified merchant names such as Swiggy, Zomato, Amazon and Uber from the raw Description column using a simple keyword-based method.

Next, I performed **category mapping**, where vendors were grouped into categories such as Food Delivery, E-commerce, Quick Commerce, Transport and Subscriptions. After preparing the data, I analysed **total credits, total debits, top categories, top vendors and monthly spending trends**.

In the final part of the project, I performed **time-of-day analysis**, **anomaly detection** and **spending archetype detection** to identify behavioural patterns such as late-night food spending, unusually high transactions and spending personality labels like Foodie or Investor.

Overall, this project helped me understand how raw financial transaction data can be cleaned, analysed and converted into useful personal finance insights using Python, pandas and NumPy.

## Resume / CV Project Points

### Resume Version 1 – Simple
- Built **SpendDNA**, a Python-based transaction analytics project to analyse six months of bank and UPI transaction data.
- Performed data cleaning, vendor extraction, category mapping, monthly spending analysis and anomaly detection using **Python, Pandas and NumPy**.
- Generated category-wise, vendor-wise and time-based spending insights from raw financial transaction records.

### Resume Version 2 – Slightly Stronger
- Developed a transaction analytics notebook to study personal spending behaviour from raw bank and UPI data.
- Cleaned and standardised transaction records, extracted vendors from payment descriptions and mapped transactions into spending categories.
- Analysed monthly trends, high-spend categories, unusual transactions and spending archetypes using Python-based data analysis.

## Resume / CV Project Points

### Resume Version 1 – Simple
- Built **SpendDNA**, a Python-based transaction analytics project to analyse six months of bank and UPI transaction data.
- Performed data cleaning, vendor extraction, category mapping, monthly spending analysis and anomaly detection using **Python, Pandas and NumPy**.
- Generated category-wise, vendor-wise and time-based spending insights from raw financial transaction records.

### Resume Version 2 – Slightly Stronger
- Developed a transaction analytics notebook to study personal spending behaviour from raw bank and UPI data.
- Cleaned and standardised transaction records, extracted vendors from payment descriptions and mapped transactions into spending categories.
- Analysed monthly trends, high-spend categories, unusual transactions and spending archetypes using Python-based data analysis.

## Project Summary for Documentation

### Project Summary:
SpendDNA is a financial transaction analytics project designed to analyse six months of bank and UPI transaction records belonging to Rahul Sharma. The main goal of the project is to convert raw and messy transaction data into meaningful spending insights. The project includes transaction data cleaning, vendor extraction from raw descriptions, category mapping, spending overview, vendor-wise and category-wise analysis, monthly spending trends, time-of-day behaviour analysis, anomaly detection and spending archetype identification. The notebook was developed using Python, Pandas and NumPy and demonstrates the use of data cleaning, grouping, aggregation and rule-based analytical logic in a real-world personal finance use case.

## Demo Explanation – What to Show While Presenting the Notebook

If I have to demonstrate this notebook in front of faculty or evaluators, I should explain the project in the following order:

### Step 1 – Introduce the problem
Explain that the project is about understanding a person's spending behaviour from raw transaction data.

### Step 2 – Show the raw dataset
Display the first few rows of the dataset and explain that the file contains Date, Time, Description, Type, Amount and other transaction-related fields.

### Step 3 – Explain data cleaning
Show the cells where Date, Amount and Type columns were cleaned and explain why cleaning is required before analysis.

### Step 4 – Explain vendor extraction
Show how merchant names were identified from the Description column using keyword matching.

### Step 5 – Explain category mapping
Show how vendors were grouped into categories such as Food Delivery, E-commerce and Transport.

### Step 6 – Show spending analysis
Display the outputs for total debit, total credit, top categories, top vendors and monthly spending.

### Step 7 – Show advanced analysis
Explain the time-of-day analysis, anomaly detection and spending archetypes.

### Step 8 – Show final report
End by showing the final SpendDNA report and the main insights from the project.

## Common Faculty Questions During Presentation

### Question 1: Why did you choose this project?
I chose this project because transaction data analysis is a practical real-world use case in personal finance and fintech. It also helped me practice data cleaning, grouping, vendor extraction and behaviour-based analysis using Python.

### Question 2: Why is vendor extraction important?
Vendor extraction is important because raw transaction descriptions are difficult to analyse directly. Vendor names make the data more readable and help in category-wise analysis.

### Question 3: Why did you use category-wise anomaly detection?
Because spending patterns are different for different categories. A large Amazon order may be normal, but the same amount in food delivery may be unusual.

### Question 4: Is the project rule-based or machine learning based?
This project is rule-based. It uses pandas operations, keyword matching, grouping and threshold-based logic rather than machine learning.

### Question 5: What would you improve if you had more time?
I would improve vendor extraction, add visualisations, create a dashboard version and build a smarter category mapping system for unknown vendors.

## Project Strengths

The main strengths of this project are:

- it works on a realistic financial dataset instead of a toy dataset
- it includes both data cleaning and analysis
- it converts raw transaction descriptions into meaningful categories
- it performs both summary analysis and behavioural analysis
- it produces a final report instead of only intermediate calculations

## Final Self-Introduction Line for Project Review

If I need to introduce this project in one or two lines during a review, I can say:

**"I developed SpendDNA, a Python-based transaction analytics project that studies six months of bank and UPI transactions to understand spending behaviour through data cleaning, vendor extraction, category mapping, monthly trend analysis and anomaly detection."**

## Final Closing Statement for Presentation

### Closing Statement:
To conclude, SpendDNA is a simple but practical transaction analytics project that demonstrates how raw financial data can be cleaned, structured and analysed to generate useful insights about personal spending behaviour. Through this project, I was able to apply Python, pandas and analytical thinking to a realistic fintech-style problem.

# Part 8 – Final Submission Wrap-Up

This final section is added to close the SpendDNA minor project in a proper academic format.  
The purpose of this section is to summarise the complete work done in the notebook and provide a final project-ending statement before submission.

The SpendDNA project was carried out in multiple stages:

1. **Data Loading and Cleaning** – the raw transaction dataset was loaded and important columns such as Date, Amount and Type were cleaned and standardised.
2. **Vendor Extraction and Category Mapping** – raw transaction descriptions were converted into meaningful vendor names and then grouped into spending categories.
3. **Spending Analysis** – category-wise, vendor-wise and monthly spending patterns were analysed.
4. **Behavioural Analysis** – time-based analysis, anomaly detection and spending archetype identification were performed.
5. **Final Interpretation** – the results were converted into insights, limitations, future scope and a final SpendDNA report.

This notebook represents the complete workflow of a transaction analytics mini project using Python, pandas and NumPy.

## Final Summary of Work Done

In this project, Rahul Sharma's six-month bank and UPI transaction data was analysed to understand his financial behaviour. The raw dataset was first cleaned to make it suitable for analysis. After that, vendor extraction and category mapping were performed to convert raw descriptions into structured information.

Once the data was prepared, the project focused on:
- total debit and total credit analysis
- category-wise and vendor-wise spending analysis
- month-wise spending trend analysis
- time-of-day behaviour analysis
- anomaly detection for unusual transactions
- spending archetype detection
- final reporting and interpretation

The project shows how a raw CSV file containing transaction records can be transformed into a meaningful financial behaviour report through data cleaning, grouping, aggregation and rule-based logic.

## Learning Outcomes from This Project

This project helped me improve my understanding of the following concepts:

- loading and inspecting CSV data using pandas
- cleaning date, amount and transaction type columns
- handling missing values and duplicate records
- using string matching for vendor extraction
- mapping transactions into business-friendly categories
- performing groupby-based spending analysis
- calculating monthly trends and category totals
- identifying unusual transactions using z-score logic
- converting numerical outputs into simple business insights

Overall, the project improved both my technical understanding of pandas operations and my practical understanding of transaction data analysis.

## Final Declaration

 The work includes data cleaning, transaction analysis, vendor extraction, category mapping and behaviour-based analysis performed on the given transaction dataset.

I used Python, Pandas and NumPy to implement the project workflow. AI tools were used only for support in understanding, debugging and improving structure wherever necessary

## Final Closing Note

The SpendDNA project is a simple but practical example of how raw financial transaction data can be converted into useful insights through data analysis. Instead of treating the dataset only as rows and columns, this project attempts to tell a spending story by combining cleaning, categorisation, trend analysis and behaviour-based interpretation.

Working on this notebook helped me understand that real-world data analysis is not only about applying functions, but also about handling messy data, designing meaningful rules and presenting the final results in a clear and understandable way.

In [117]:
print("=" * 70)
print("        SPENDDNA MINOR PROJECT - FINAL SUBMISSION NOTE")
print("=" * 70)
print("Project completed successfully.")
print("Student Name : Pradhapkumar S")
print("Project Name : SpendDNA - Wallet Story Analysis")
print("Tools Used   : Python, Pandas, NumPy")
print("Dataset Used : Rahul Sharma Transaction Dataset")
print("Status       : Ready for submission")
print("=" * 70)

        SPENDDNA MINOR PROJECT - FINAL SUBMISSION NOTE
Project completed successfully.
Student Name : Pradhapkumar S
Project Name : SpendDNA - Wallet Story Analysis
Tools Used   : Python, Pandas, NumPy
Dataset Used : Rahul Sharma Transaction Dataset
Status       : Ready for submission


# Thank You
 The completion of the **SpendDNA Minor Project**.  
The project was developed to understand Rahul Sharma's spending behaviour using transaction data analysis in Python.

# Thank You

The completion of the **SpendDNA Minor Project**.  
The project was developed to understand Rahul Sharma's spending behaviour using transaction data analysis in Python.

